# DoubleBasis basics

This notebook introduces `DoubleBasis`, the object EDKit uses to describe a map between two bases. The main point is to separate two ideas:

- `DoubleBasis(Btarget, Bsource)` stores the metadata for a linear map from coordinates in `Bsource` to coordinates in `Btarget`.
- `symmetrizer(T)` constructs that map explicitly as a matrix.
- `T(v)` applies the same map directly, without first building the dense matrix.

We start with an `AbelianBasis` target, because that is now the most general symmetry-facing basis in the package.

In [ ]:
import Pkg
using LinearAlgebra

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
    include(joinpath(REPO_ROOT, "src", "EDKit.jl"))
    using .EDKit
else
    using EDKit
end


## Build a target sector and a source basis

We work with a spin-1/2 chain of length `L = 4`. The source basis is the full tensor-product basis. The target basis is a symmetry-resolved `AbelianBasis` built with the high-level `basis(...)` helper: half filling, zero momentum, and even parity.

This is the natural situation for a symmetrizer: start from the full Hilbert space and map into a smaller symmetry sector.

In [ ]:
L = 4
Bsource = TensorBasis(L = L, base = 2)
Btarget = basis(L = L, N = 2, k = 0, p = 1, base = 2, threaded = false)
T = DoubleBasis(Btarget, Bsource)

summary_bases = (
    source_dimension = size(Bsource, 1),
    target_dimension = size(Btarget, 1),
    map_size = size(T),
)
summary_bases


## Direction of the map

The order matters: `DoubleBasis(Btarget, Bsource)` means the map acts on a vector written in `Bsource` coordinates and returns a vector written in `Btarget` coordinates.

So if `psi_full` lives in the full Hilbert space, `T(psi_full)` gives the symmetry-sector coordinates.

In [ ]:
psi_full = randn(ComplexF64, size(Bsource, 1)) |> normalize
psi_sector = T(psi_full)

summary_action = (
    input_length = length(psi_full),
    output_length = length(psi_sector),
    output_norm = norm(psi_sector),
)
summary_action


## Fast action versus explicit matrix

The direct call `T(v)` should agree with the explicit matrix action `symmetrizer(T) * v`. The point of the callable form is to avoid the cost of materializing the whole map when we only need to apply it once or a few times.

In [ ]:
S = symmetrizer(T)
psi_sector_matrix = S * psi_full

summary_compare = (
    explicit_matrix_size = size(S),
    fast_vs_matrix_error = norm(psi_sector - psi_sector_matrix),
)
summary_compare


## A legacy basis still works

Even though `AbelianBasis` is the general path, the same interface should still work for the older specialized bases. Here we check the parity basis against the full tensor basis and verify the same fast-action identity.

In [ ]:
Bpar = ParityBasis(L = L, p = 1, base = 2, threaded = false)
Tpar = DoubleBasis(Bpar, Bsource)
psi_par_fast = Tpar(psi_full)
psi_par_matrix = symmetrizer(Tpar) * psi_full

summary_legacy = (
    parity_dimension = size(Bpar, 1),
    parity_fast_vs_matrix_error = norm(psi_par_fast - psi_par_matrix),
)
summary_legacy


## Key takeaway

`DoubleBasis` is not itself an operator term like a Hamiltonian. It is metadata describing the domain and codomain of a basis map. The special map associated with it is `symmetrizer(T)`, and the fast application path is `T(v)`. Once that picture is clear, non-square operators become much easier to interpret.